# Fine-tuning Gemma 1b

On the raw conversations dataset.

Reference: [Gemma3N_(4B)-Conversational](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma3N_(4B)-Conversational.ipynb).

In [1]:
dataset_path = "../output/dataset/dspy-dataset.csv"
model_output_path = "../output/models/gemma-3-1b-raw"
max_seq_length = 2048
save_merged = False

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it",
    max_seq_length=max_seq_length,
    dtype=None,
)

==((====))==  Unsloth 2025.7.8: Fast Gemma3 patching. Transformers: 4.53.3. vLLM: 0.8.3.
   \\   /|    NVIDIA RTX 2000 Ada Generation Laptop GPU. Num GPUs = 1. Max memory: 7.747 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [25]:
model = FastLanguageModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=8,
    lora_dropout=0,
    random_state=7,
)

Unsloth: Making `model.base_model.model.model` require gradients


In [47]:
import pandas as pd
import json
from datasets import Dataset


def load_dataset():
    messages = []

    df = pd.read_csv(dataset_path, sep=",", encoding="utf-8")
    for _, row in df.iterrows():
        messages.append(json.loads(row['conversations']))

    res = Dataset.from_dict({"conversations": messages})
    return res

dataset = load_dataset()
dataset

Dataset({
    features: ['conversations'],
    num_rows: 229
})

In [48]:
dataset[0]

{'conversations': [{'content': 'Hai paura di affrontarmi? Fatti vedere!\nVeramente, mi stai già guardando!',
   'role': 'human'},
  {'content': 'Io sono questo edificio!', 'role': 'assistant'},
  {'content': 'Sglurg!', 'role': 'human'},
  {'content': 'Ma se proprio hai bisogno di un volto a cui rivolgerti ... ecco!',
   'role': 'assistant'},
  {'content': 'Glom! Penso di averne viste abbastanza, per una sola notte!\nFacciamoci coraggio!\nNon so cosa tu sia, ma hai di fronte Paperi-Nik!',
   'role': 'human'},
  {'content': 'Piacere! Io sono uno!', 'role': 'assistant'},
  {'content': 'Uno, eh? Dove hai lasciato gli altri?', 'role': 'human'},
  {'content': 'Groan! Che battutaccia! È terribile!', 'role': 'assistant'},
  {'content': 'Immagino che ti aspetti spiegazioni! Mettiti comodo!',
   'role': 'human'},
  {'content': 'Molto Gentile!', 'role': 'assistant'},
  {'content': 'Dunque sei un computer?', 'role': 'human'},
  {'content': "Un'intelligenza artificiale, prego!\nPer essere precisi, 

In [49]:
split_dataset = dataset.train_test_split(test_size=0.1, seed=442)
train_dataset = split_dataset['train']
test_dataset = split_dataset['test']
train_dataset, test_dataset

(Dataset({
     features: ['conversations'],
     num_rows: 206
 }),
 Dataset({
     features: ['conversations'],
     num_rows: 23
 }))

In [50]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-3",
)

In [51]:
from unsloth.chat_templates import standardize_data_formats

train_dataset = standardize_data_formats(train_dataset)

Unsloth: Standardizing formats (num_proc=22):   0%|          | 0/206 [00:00<?, ? examples/s]

In [52]:
train_dataset[0]

{'conversations': [{'content': "E così, ci stanno tenendo d'occhio...\nGià, ma possono fare ben poco, oltre a starsene lì tutto il giorno.\nCon i loro strumenti non riusciranno mai a superare le difese di Padron Ducklair!\nSei proprio un cattivone, uno! Perché non gli diamo un contentino?",
   'role': 'user'},
  {'content': "Quello l'hanno già avuto...", 'role': 'assistant'}]}

In [ ]:
# We remove the <bos> token using removeprefix('<bos>') since we're finetuning.
# The Processor will add this token before training and the model expects only
# one.

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix('<bos>') for convo in convos]
    return {"text": texts, }


train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/206 [00:00<?, ? examples/s]

In [55]:
train_dataset[10]["text"]

"<start_of_turn>user\nL'unica intervista esistente con la nuova star del cinema! I suoi amori presenti, passati e soprattutto futuri!\nAh! Ah! Ah! Passerai alla storia!\nNiente Evroniani! Anche questa volta è andata bene.\nChe sforzo! Ti è venuta l'ernia?\nDivina? E chi sarebbe?\nInvidiosi! Non mi meritate!<end_of_turn>\n<start_of_turn>model\nNon riesco proprio a trovare una strada per entrare.\nCosa sta succedendo?<end_of_turn>\n"

## Fine-tune

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=None,  # Can set up evaluation!
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,  # Use GA to mimic batch size!
        warmup_steps=5,
        num_train_epochs=3,
        # max_steps=60,  # Uncomment for debugging
        learning_rate=2e-4,  # Reduce to 2e-5 for long training runs
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",  # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/206 [00:00<?, ? examples/s]

In [58]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=22):   0%|          | 0/206 [00:00<?, ? examples/s]

In [59]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

"<bos><start_of_turn>user\nUh? I comandi si sono riattivati!\nAppena in tempo!<end_of_turn>\n<start_of_turn>model\nCi sei, socio? Ho novità molto strane!<end_of_turn>\n<start_of_turn>user\nMai quanto le mie! L'ufo mi ha disattivato il sistema di guida e se l'è squagliata!<end_of_turn>\n<start_of_turn>model\nCome ha fatto? I comandi della Pi-Kar sono schermati!<end_of_turn>\n<start_of_turn>user\nComunque sono incolume! Tu che cosa mi dici?<end_of_turn>\n<start_of_turn>model\nMi sono collegato con gli elaboratori della Paperopolitan Technofinancial, per vedere cos'ha combinato il tuo avversario invisibile.<end_of_turn>\n<start_of_turn>user\nCos' hai scoperto?<end_of_turn>\n<start_of_turn>model\nNulla!\nTutti i computer del palazzo sono... bruciati.<end_of_turn>\n"

In [60]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

"                          Ci sei, socio? Ho novità molto strane!<end_of_turn>\n                                   Come ha fatto? I comandi della Pi-Kar sono schermati!<end_of_turn>\n                    Mi sono collegato con gli elaboratori della Paperopolitan Technofinancial, per vedere cos'ha combinato il tuo avversario invisibile.<end_of_turn>\n             Nulla!\nTutti i computer del palazzo sono... bruciati.<end_of_turn>\n"

## Start train

In [61]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 206 | Num Epochs = 3 | Total steps = 156
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 6,522,880 of 1,006,408,832 (0.65% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1,5.516400
2,5.855700
3,5.019200
4,5.562000
5,5.313600
6,5.011400
7,4.149200
8,4.386400
9,4.055200
10,3.912100


Unsloth: Will smartly offload gradients to save VRAM!


In [62]:
import torch

# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")

298.6319 seconds used for training.
4.98 minutes used for training.
Peak reserved memory = 2.266 GB.


Run inference

In [64]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "Chi sei?",
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

['<bos><start_of_turn>user\nChi sei?<end_of_turn>\n<start_of_turn>model\nSono io! Il tuo assistente virtuale, pronto a fare doverti compagnia!<end_of_turn>']

## Saving

In [68]:
model.save_pretrained(model_output_path + "/lora")
tokenizer.save_pretrained(model_output_path + "/lora")

('../output/models/gemma-3-1b-raw/lora/tokenizer_config.json',
 '../output/models/gemma-3-1b-raw/lora/special_tokens_map.json',
 '../output/models/gemma-3-1b-raw/lora/chat_template.jinja',
 '../output/models/gemma-3-1b-raw/lora/tokenizer.model',
 '../output/models/gemma-3-1b-raw/lora/added_tokens.json',
 '../output/models/gemma-3-1b-raw/lora/tokenizer.json')

In [ ]:
if save_merged:
    model.save_pretrained_merged(
        model_output_path + "/merged",
        tokenizer=tokenizer
    )

Found HuggingFace hub cache directory: /home/mbrt/.cache/huggingface/hub
Checking cache directory for required files...
Successfully copied all 1 files from cache to ../output/models/gemma-3-1b-raw/merged.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:03<00:00,  3.52s/it]


In [ ]:
if save_merged:
    model.save_pretrained_gguf(
        model_output_path + "/merged",
        quantization_type="Q8_0",
    )

Unsloth: Updating system package directories
Unsloth: All commands will now use admin permissions (sudo)
Unsloth: Install GGUF and other packages
Unsloth GGUF:hf-to-gguf:Loading model: merged
Unsloth GGUF:hf-to-gguf:Model architecture: Gemma3ForCausalLM
Unsloth GGUF:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
Unsloth GGUF:hf-to-gguf:Exporting model...
Unsloth GGUF:hf-to-gguf:gguf: loading model part 'model.safetensors'
Unsloth GGUF:hf-to-gguf:token_embd.weight,                 torch.bfloat16 --> Q8_0, shape = {1152, 262144}
Unsloth GGUF:hf-to-gguf:output_norm.weight,                torch.bfloat16 --> F32, shape = {1152}
Unsloth GGUF:hf-to-gguf:Set meta model
Unsloth GGUF:hf-to-gguf:Set model parameters
Unsloth GGUF:hf-to-gguf:Set model quantization version
Unsloth GGUF:hf-to-gguf:Set model tokenizer
Unsloth GGUF:gguf.vocab:Setting special token type bos to 2
Unsloth GGUF:gguf.vocab:Setting special token type eos to 106
Unsloth GGUF:gguf.vocab:Setting special token t

Unsloth: GGUF conversion:   0%|          | 0/100 [00:00<?, ?it/s]

Unsloth GGUF:hf-to-gguf:Model successfully exported to ../output/models/gemma-3-1b-raw/
Unsloth: Converted to ../output/models/gemma-3-1b-raw/merged.Q8_0.gguf with size = 1.1G
Unsloth: Successfully saved GGUF to:
../output/models/gemma-3-1b-raw/merged.Q8_0.gguf


['../output/models/gemma-3-1b-raw/merged.Q8_0.gguf']

Save to Ollama: [Reference](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_output_path + "/lora",
    max_seq_length=max_seq_length,
)
model = FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2025.7.8: Fast Gemma3 patching. Transformers: 4.53.3. vLLM: 0.8.3.
   \\   /|    NVIDIA RTX 2000 Ada Generation Laptop GPU. Num GPUs = 1. Max memory: 7.747 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: We automatically added an EOS token to stop endless generations.


AttributeError: 'NoneType' object has no attribute 'map'

In [ ]:
# TODO: fix
# Seems like we need to apply the chat template again
# https://github.com/unslothai/unsloth/issues/798

model.save_pretrained_gguf(
    model_output_path + "/gguf",
    tokenizer,
)

TypeError: Unsloth: quantization_method can only be a string or a list of strings